## **Goal**
**Given the ground truth (Unreal Engine) coordinates and camera rotation vector, calculate:**
* Pointing vector (shoulder joint -> wrist joint) in camera coordinates
* Distance from shoulder joint to camera
* Angles between shoulder-wrist and shoulder-camera vectors

### **Setup**

```
conda create -n fbv-gt-calc python=3.10
conda activate fbv-gt-calc
conda install numpy
conda install ipykernel=6.29.5
```

In [3]:
import numpy as np
import json

### **Given**

In [4]:
# Load JSON Metadata

JSON_A_PATH = "../../../dataset/metadata_A/00003A.json"

with open(JSON_A_PATH, 'r') as file:
    raw_data = json.load(file)

data = {
    'RS': np.array([raw_data['Right Shoulder Coords'][axis] for axis in ['x', 'y', 'z']]),
    'RW': np.array([raw_data['Right Wrist Coords'][axis] for axis in ['x', 'y', 'z']]),
    'CC': np.array([raw_data['Camera Coords'][axis] for axis in ['x', 'y', 'z']]),
    'CR': np.array([raw_data['Camera Rotator'][axis] for axis in ['x', 'y', 'z']]),
}

print(data)

{'RS': array([-4581.53774476,  -185.47113061,   141.47604939]), 'RW': array([-4528.30575964,  -185.76802202,   141.55266454]), 'CC': array([-4397.82601405,  -369.18286131,   291.47604939]), 'CR': array([-0.61237244,  0.61237244, -0.5       ])}


In [5]:
# # Keypoint Global Coordinates

# shoulder_coords_global = np.array([-4581.538, -185.471, 141.476])
# wrist_coords_global = np.array([-4528.306, -185.768, 141.553])
# camera_coords_global = np.array([-4079.628, -319.957, 441.476])
# camera_forward_global = np.array([-0.837, 0.224, -0.500])

In [6]:
# # Keypoint Global Coordinates

# shoulder_coords_global = np.array([-4581.538 ,-185.471 ,141.47])
# wrist_coords_global = np.array([-4531.860 ,-183.063 ,122.502])
# camera_coords_global = np.array([-4581.538 ,414.529 ,141.476])
# camera_forward_global = np.array([0.000 ,-1.000 ,0.000])

In [7]:
# Keypoint Global Coordinates

shoulder_coords_global = data['RS']
wrist_coords_global = data['RW']
camera_coords_global = data['CC']
camera_forward_global = data['CR']

In [8]:
# Assumed Constants

camera_coords = np.array([0, 0, 0])
world_up = np.array([0, 0, 1])

In [9]:
# Expected Results

expected_distance = 600.0 # cm
expected_yaw = -15.0      # deg
expected_pitch = 30.0     # deg
expected_angular_separation = np.sqrt(expected_yaw**2 + expected_pitch**2)

### **Calculations**

In [10]:
# # Create Rotation Matrix

# camera_right_global = np.cross(camera_forward_global, world_up)
# camera_right_global /= np.linalg.norm(camera_right_global)

# camera_up_global = np.cross(camera_forward_global, camera_right_global)
# camera_up_global /= np.linalg.norm(camera_up_global)

# rotation_matrix = np.vstack((camera_forward_global,
#                              camera_right_global,
#                              camera_up_global))

# print(rotation_matrix)

In [11]:
# Create Rotation Matrix

camera_right_global = np.cross(world_up, camera_forward_global) 
camera_right_global /= np.linalg.norm(camera_right_global)

camera_up_global = np.cross(camera_forward_global, camera_right_global)
camera_up_global /= np.linalg.norm(camera_up_global)

rotation_matrix = np.vstack((camera_forward_global,
                             camera_right_global,
                             camera_up_global))
print(rotation_matrix)

[[-0.61237244  0.61237244 -0.5       ]
 [-0.70710678 -0.70710678  0.        ]
 [-0.35355339  0.35355339  0.8660254 ]]


In [12]:
# Translate To Origin

shoulder_coords = shoulder_coords_global - camera_coords_global
wrist_coords = wrist_coords_global - camera_coords_global

print(shoulder_coords)
print(wrist_coords)

[-183.71173071  183.71173071 -150.        ]
[-130.47974559  183.41483929 -149.92338485]


In [13]:
# Apply Rotation Transformation

shoulder_coords = rotation_matrix @ shoulder_coords
wrist_coords = rotation_matrix @ wrist_coords

print(shoulder_coords)
print(wrist_coords)

[ 3.00000000e+02  8.52651283e-14 -5.68434189e-14]
[267.18208392 -37.43076371 -18.85896512]


### **Analysis**

In [14]:
# Shoulder-Camera Distance

distance = np.linalg.norm(shoulder_coords)
print(f"Distance should be close to {expected_distance} cm:")
print(distance, "cm")

Distance should be close to 600.0 cm:
300.0 cm


In [15]:
# Shoulder-Wrist Shoulder-Camera Vectors

shoulder_wrist = wrist_coords - shoulder_coords
shoulder_wrist /= np.linalg.norm(shoulder_wrist)

shoulder_camera = camera_coords - shoulder_coords
shoulder_camera /= np.linalg.norm(shoulder_camera)

print(shoulder_wrist)
print(shoulder_camera)

[-0.61649724 -0.70315136 -0.35427295]
[-1.00000000e+00 -2.84217094e-16  1.89478063e-16]


In [16]:
# Angular Separation

angular_separation_rad = np.arccos(np.clip(np.dot(shoulder_wrist, shoulder_camera), -1.0, 1.0))
angular_separation_deg = np.rad2deg(angular_separation_rad)

print(f"Angular separation should be close to {expected_angular_separation} deg:")
print(angular_separation_deg)


Angular separation should be close to 33.54101966249684 deg:
51.939207159221475


### **Angular Separation Components**
Given the angular separation of the shoulder-wrist and shoulder-camera vectors, and camera pitch angle (which could be
retreived from an onboard gyroscope), calculate the vertical component (pitch/elevation) and the horizontal component
(yaw/azimuth) of the angular separation in global coordinates.

In [17]:
# Camera Pitch (Ground Truth / Gyroscope)

camera_pitch_rad = np.asin(camera_forward_global[-1])
camera_pitch_deg = np.rad2deg(camera_pitch_rad)
print(f"Camera pitch should be close to {expected_pitch} deg:")
print(camera_pitch_deg, "deg")

Camera pitch should be close to 30.0 deg:
-29.999999999999993 deg


In [18]:
# # Camera Un-Pitch Matrix (Aligns Z-Axis with Gravity)

# unpitch_matrix = np.array([[np.cos(-camera_pitch_rad), 0, np.sin(-camera_pitch_rad)],
#                          [0, 1, 0],
#                          [-np.sin(-camera_pitch_rad), 0, np.cos(-camera_pitch_rad)]])

# print(unpitch_matrix)

In [19]:
# Camera Un-Pitch Matrix (Aligns Z-Axis with Gravity)

c, s = np.cos(-camera_pitch_rad), np.sin(-camera_pitch_rad)
unpitch_matrix = np.array([
    [c,  0, s],
    [0,  1, 0],
    [-s, 0, c]
])
print(unpitch_matrix)

[[ 0.8660254  0.         0.5      ]
 [ 0.         1.         0.       ]
 [-0.5        0.         0.8660254]]


In [20]:
# Un-Pitch Vectors

shoulder_wrist_gravity = unpitch_matrix @ shoulder_wrist
shoulder_wrist_gravity /= np.linalg.norm(shoulder_wrist_gravity)

shoulder_camera_gravity = unpitch_matrix @ shoulder_camera
shoulder_camera_gravity /= np.linalg.norm(shoulder_camera_gravity)

print(shoulder_wrist_gravity)
print(shoulder_camera_gravity)

[-0.71103874 -0.70315136  0.00143925]
[-8.66025404e-01 -2.84217094e-16  5.00000000e-01]


In [21]:
# Yaw & Pitch Components

shoulder_wrist_gravity_yaw = np.rad2deg(np.atan2(shoulder_wrist_gravity[1], shoulder_wrist_gravity[0]))
shoulder_wrist_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_wrist_gravity[-1], -1.0, 1.0)))

shoulder_camera_gravity_yaw = np.rad2deg(np.atan2(shoulder_camera_gravity[1], shoulder_camera_gravity[0]))
shoulder_camera_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_camera_gravity[-1], -1.0, 1.0)))

delta_yaw = (shoulder_wrist_gravity_yaw - shoulder_camera_gravity_yaw + 180) % 360 - 180
delta_pitch = shoulder_wrist_gravity_pitch - shoulder_camera_gravity_pitch

print(delta_yaw)
print(delta_pitch)

44.680446842088486
-29.917537292217368
